In [ ]:
# -*- coding: utf-8 -*-
"""

Classes and Functions of the SCM project for SMART Research Group
Luis Lamas - 19/aug/2021

"""

import win32com.client
import numpy as np
import re
#import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import integrate




class iPhreeqc_model:
    def __init__(self, DatabasePath, MasterFilePath):
        """
		Initialize the iPhreeqc_model

		Parameters:
		  DatabasePath : Path to the database file
        Returns None

        Created by Luis Lamas, 19/aug/2021
		"""
        self.DatabasePath = DatabasePath
        self.MasterFilePath = MasterFilePath
        self.MasterFile = None
        self.MasterParameters = None
        self.ServerStatus = 'Not Initialized'
        self.iPhreeqcInstance = None
        
        self.ReadMasterFile()


    def InitializeServer(self):
        """
		Initialize the iPhreeqc_model

		Parameters:
		  self object
        Returns None
        Updates the variable self.ServerStatus

        Created by Luis Lamas, 19/aug/2021
		"""
        try:
            iphreeqc = win32com.client.Dispatch("IPhreeqcCOM.Object")
            iphreeqc.LoadDatabase(self.DatabasePath)
            self.ServerStatus = 'Server Initizlized'
            self.iPhreeqcInstance = iphreeqc
        except:
            self.ServerStatus = 'Error to initialize Server'
        
        

    def ReadMasterFile(self):
        """
		Reads the Master File

		Parameters:
		  self object
        Returns None
        Updates the variable self.MasterFile        
        
        Created by Luis Lamas, 19/aug/2021        
		"""        
        with open(self.MasterFilePath, 'r') as file:
            self.MasterFile = file.readlines()
            
        
    def GetMasterParameters(self):
        """
		Gets parameters from Master file

		Parameters:
		  self object
        Returns list of parameters found in MasterFile
        Updates the variable self.MasterParameters     
        
        Created by Luis Lamas, 19/aug/2021        
		"""                
        MasterFile = self.MasterFile
        Params = []
        for line in MasterFile:
            result = re.search('<<ALTER(.*)>>', line)
            if result != None:
                aux_txt = result.group(1).split('=')
                var = aux_txt[-1].strip()
                Params.append(var)
        self.MasterParameters = Params
        return Params
        

    def CreateSimulationFile(self, ParameterDictionary):
        """
		Gets parameters from Master file

		Parameters:
		  self object
          Params: Dictionary containing the values to be replaced in the simulation file
        Returns list of lines to be inserted in the simulation
        
        Created by Luis Lamas, 19/aug/2021        
		"""                
        SimulationFile = []
        if self.MasterFile is None:
            self.ReadMasterFile()
        MasterFile = self.MasterFile
        for line in MasterFile:
            result = re.findall(r'<<ALTER, variable = (\w+)>>', line)
            for txt in result:
                aux_txt = f'<<ALTER, variable = {txt}>>'
                value = ParameterDictionary[txt]
                line = re.sub(aux_txt, str(value), line, flags=re.DOTALL)            
            SimulationFile.append(line)
        return SimulationFile


    def RunPhreeqc(self, ParameterDictionary):
        """
		Gets parameters from Master file

		Parameters:
		  self object
          Params: Dictionary containing the values to be replaced in the simulation file
        Returns list of lines to be inserted in the simulation
        
        Created by Luis Lamas, 19/aug/2021        
		"""                
        if self.ServerStatus == 'Not Initialized':
            self.InitializeServer()
            if self.ServerStatus == 'Error to initialize Server':
                print('Error in the initialization of Server')
        
        SimFile = self.CreateSimulationFile(ParameterDictionary)
        
        iphreeqc = self.iPhreeqcInstance
        
        SimFile = "\n".join(SimFile)
        
        iphreeqc.OutputStringOn = True
        
        try:
            runresult = iphreeqc.RunString(SimFile)
        except:
            pass
    
        try:
            out_PHREEQC = iphreeqc.GetSelectedOutputArray()
        except:
            out_PHREEQC = -9999
        return out_PHREEQC, iphreeqc 



class SCM_model:
    def __init__(self, iphreeqc):
        """
		Initialize the iPhreeqc_model

		Parameters:
		  DatabasePath : Path to the database file
        Returns None

        Created by Luis Lamas, 19/aug/2021
		"""
        self.iphreeqc = iphreeqc
        self.Temperature = 25+273.15    # K
        self.eps_w = 78.3    # Water dielectric constant
        self.eps_r = 7       # Rock dielectric constant
        self.eps_o = 2       # Oil dielectric constant
        self.nw = 1.33       # Water refractive index
        self.nr = 1.5        # Rock refractive index
        self.no = 1.4        # Oil refractive index        
        self.epsr = 80
        self.ni_e = 3e15     # 1/s 
        self.sigma = 25      # mN/m
        self.Pc = 0          # kPa

        self.EDL_Calcite = []
        self.EDL_Oil = []
        self.IonicStrength = []
        
        self.GetPhreeqcParams(iphreeqc)



    def PhysicalConstants(self, arg):
        ConstDict = {'kb': 1.38064852e-23,    # m2 kg s-2 K-1   Boltzmann Constant
                      'e': 1.60217662e-19,    # C               Electron charge
                      'NA': 6.0221409e+23,      # mol-1          Avogrado number
                      'eps0': 8.854e-12,       # Cˆ2/J.m        Vacuum permittivity
                      'ni_e': 3e15,            # 1/s            Electrical adsorption frequency
                      'h_plank': 6.626e-34}    # J.s   10 34     Plank constant
    
                     
        return ConstDict[arg]
    


    def GetOilComponents(self):
        out_PHREEQC = self.iphreeqc.GetSelectedOutputArray()
        
        idx_oil = out_PHREEQC[0].index('V_EDL_Oil')
        for i in range(1, len(out_PHREEQC)):
            if out_PHREEQC[i][idx_oil] is not None:
                line_oil = i
        
        out_PHREEQC_Oil = out_PHREEQC[line_oil]
        dummy_positions = [i for i, string in enumerate(out_PHREEQC[0]) if string.find('no_heading') == -1]
        out_PHREEQC_Oil = np.array(out_PHREEQC_Oil)
        out_PHREEQC_Oil[dummy_positions] = None
        out_PHREEQC_Oil = list(filter(None, out_PHREEQC_Oil))
        out_PHREEQC_Oil = np.reshape(out_PHREEQC_Oil, (int(len(out_PHREEQC_Oil)/2), 2))
        
        Oil_components = [s for s in out_PHREEQC_Oil[:,0] if "Oil_" in s]  
        Oil_concentrations = [10**float(s) for s in out_PHREEQC_Oil[:,1]]

        return Oil_components, Oil_concentrations


    def GetRockComponents(self):
        out_PHREEQC = self.iphreeqc.GetSelectedOutputArray()
        
        idx_calcite = out_PHREEQC[0].index('V_EDL_Calcite')
        for i in range(1, len(out_PHREEQC)):
            if out_PHREEQC[i][idx_calcite] is not None:
                line_calcite = i

        dummy_positions = [i for i, string in enumerate(out_PHREEQC[0]) if string.find('no_heading') == -1]
        out_PHREEQC_Calcite = out_PHREEQC[line_calcite]
        out_PHREEQC_Calcite = np.array(out_PHREEQC_Calcite)
        out_PHREEQC_Calcite[dummy_positions] = None
        out_PHREEQC_Calcite = list(filter(None, out_PHREEQC_Calcite))
        out_PHREEQC_Calcite = np.reshape(out_PHREEQC_Calcite, (int(len(out_PHREEQC_Calcite)/2), 2))        

        Calcite_components = [s for s in out_PHREEQC_Calcite[:,0] if "Calcite_" in s]
        Calcite_concentrations = [10**float(s) for s in out_PHREEQC_Calcite[:,1]]      
        
        return Calcite_components, Calcite_concentrations

    def CalculateBPS(self):
        """
    		Calculates BPS and individual bonds
    
    		Parameters:
    		  out_PHREEQC: Output from previous iPhreeqc run
    
        Returns value of BPS and list with bonds
        
        Created by Luis Lamas, 23/aug/2021        
     	"""                    

        Oil_components, Oil_concentrations = self.GetOilComponents()
        Calcite_components, Calcite_concentrations = self.GetRockComponents()
        
        bond = []
        for i in range(len(Oil_components)):
            aux_oil = Oil_components[i]
            
            if aux_oil.find('-') != -1:    # Componente do óleo negativo
                for j in range(len(Calcite_components)):
                    aux_calcite = Calcite_components[j]
                    if aux_calcite.find('+') != -1:
                        bond.append(Calcite_components[j])
                        bond.append(Oil_components[i])
        
            if aux_oil.find('+') != -1:    # Componente do óleo negativo
                for j in range(len(Calcite_components)):
                    aux_calcite = Calcite_components[j]
                    if aux_calcite.find('-') != -1:
                        bond.append(Calcite_components[j])
                        bond.append(Oil_components[i])
        
        bond = np.reshape(bond, (np.int(len(bond)/2), 2))    
        
        
        bond_product_output = []
        BPS = 0
        
        for j in range(np.size(bond,0)):
            molality_calcite = Calcite_concentrations[Calcite_components.index(bond[j,0])]
            molality_oil = Oil_concentrations[Oil_components.index(bond[j,1])]
            
            fraction_calcite = molality_calcite # / molality_master_calcite
            fraction_oil = molality_oil # / molality_master_oil
            
            bond_product = fraction_calcite * fraction_oil
            bond_product_output.append(bond_product)
            
            BPS = BPS + bond_product
        
    
        return BPS, (bond, bond_product_output)
    

    def PI_Struct(self, h):
        """
    		Calculates Structural Forces
    
    		Parameters:
    		  h: Distance between plates (m)
    
        Returns value structural force (kPa)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""       
        As = 1.5e10/1e3    # kPa
        hs = 0.05e-9       # m
        
        StrucForces = As * np.exp(-h/hs)
        
        return StrucForces
    
    
    def PI_vdW(self, h):
        """
    		Calculates van der Waals Forces
    
    		Parameters:
    		  h: Distance between plates (m)
    
        Returns van der Waals force (kPa)
        
        ** TODO: Put all input arguments and physical constants as input (maybe external file or global variable)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""          
        eps_w = self.eps_w
        eps_r = self.eps_r
        eps_o = self.eps_o
        nw = self.nw
        nr = self.nr
        no = self.no
        T = self.Temperature
        
        kb = self.PhysicalConstants('kb')
        ni_e = self.PhysicalConstants('ni_e')
        h_plank =self. PhysicalConstants('h_plank')
        
        A1 = (3/4)*(kb*T)*((eps_o-eps_w)/(eps_o+eps_w))*((eps_r-eps_w)/(eps_r+eps_w))
        aux1 = (3*h_plank*ni_e/(8*np.sqrt(2)))
        aux2 = (no**2-nw**2)*(nr**2-nw**2)
        aux3 = (no**2+nw**2)**0.5 * (nr**2+nw**2)**0.5
        aux4 = (no**2+nw**2)**0.5 + (nr**2+nw**2)**0.5
        A2 = aux1*aux2/(aux3*aux4)
    
        kappa = self.DebbyeLength()
#        kappa = 1/k_inv
        
        A = A1*(2*h/kappa)*np.exp(-2*h/kappa) + A2    ## Bordeaux
        A = A1 + A2    ## Sanaei
        
        
        vdW = -A/(6*np.pi*h**3)
        vdW = vdW / 1e3
        
        return vdW
    

    def PI_EDL(self, h):
        """
    		Calculates Electrical Double Layer Forces
    
    		Parameters:
    		  h: Distance between plates (m)
              psi1 and psi2: Electrical potential of the plates (V)
              I: Ionic strenght of the solution (mol/kgw)
    
        Returns EDL force (kPa)
        
        ** TODO: Put all input arguments and physical constants as input (maybe external file or global variable)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""              
        eps0 = self.PhysicalConstants('eps0')
        epsr = self.epsr
        
        psi1 = self.EDL_Calcite
        psi2 = self.EDL_Oil
    
        kappa = self.DebbyeLength()
        
        EDL = eps0*epsr*((1/kappa)**2)*(2*psi1*psi2*np.cosh(h/kappa)-(psi1**2+psi2**2))/(2*(np.sinh(h/kappa))**2)/1e3
        
        return EDL
    



    def GetPhreeqcParams(self, iphreeqc):
        """
    		Retrieves Data for EDL and Contact Angle Calculation
    
    		Parameters:
              out_PHREEQC: list of outputs of Selected Output in Phreeqc
    
            Returns:
                EDL_Calcite (V)
                EDL_Oil (V)
                I: Ionic strenght of the solution (mol/m3)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""                  
         
        self.iphreeqc = iphreeqc
        out_PHREEQC = self.iphreeqc.GetSelectedOutputArray()         
        
        try:
            idx_calcite = out_PHREEQC[0].index('V_EDL_Calcite')
        except:
            idx_calcite = 0
        try:
            idx_oil = out_PHREEQC[0].index('V_EDL_Oil')
        except:
            idx_oil = 0
            
        line_calcite = None
        line_oil = None
        for i in range(1, len(out_PHREEQC)):
            if (out_PHREEQC[i][idx_calcite] is not None) & (line_calcite is None):
                line_calcite = i
            if (out_PHREEQC[i][idx_oil] is not None) & (line_oil is None):
                line_oil = i
    
        if idx_calcite != 0:
            EDL_Calcite = out_PHREEQC[line_calcite][out_PHREEQC[0].index('V_EDL_Calcite')]
        else:
            EDL_Calcite = -9999

        if idx_oil != 0:
            EDL_Oil = out_PHREEQC[line_oil][out_PHREEQC[0].index('V_EDL_Oil')]
        else:
            EDL_Oil = -9999
        I = 1000*out_PHREEQC[line_calcite][out_PHREEQC[0].index('mu')]    
        
        T = out_PHREEQC[line_calcite][out_PHREEQC[0].index('temp(C)')]  + 273.15

        self.EDL_Calcite = EDL_Calcite
        self.EDL_Oil = EDL_Oil
        self.IonicStrength = I
        self.Temperature = T
        
#        return EDL_Calcite, EDL_Oil, I



    def PI_DLVO(self, h):
        """
    		Calculates Electrical Double Layer Forces
    
    		Parameters:
    		  h: Distance between plates (nm)
              psi1 and psi2: Electrical potential of the plates (V)
              I: Ionic strenght of the solution (mol/kgw)
    
        Returns Disjouning Pressure (kPa)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""              
        
        StrucForces = self.PI_Struct(h)
        vdW = self.PI_vdW(h)    
        EDL = self.PI_EDL(h)
        
        DLVO = StrucForces + vdW + EDL
        
        return DLVO
        


    def ContactAngle(self, h0):
        """
    		Calculates Contact Angle
    
    		Parameters:
              h0: Equilibrium distance with cappilary pressure (nm)
              hmax: Maximum distance to calculate integral (nm)
              sigma: Interfacial tension (mN/m)
              psi1 and psi2: Electrical potential of the plates (V)
              I: Ionic strenght of the solution (mol/kgw)              
    
    
        Returns contact angle (degrees)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""                  
        
        sigma = self.sigma
        
#        h0 = self.CalculateH0()
        Pc = self.Pc
        
        hmax = 100e-9

        func_DLVO = lambda x: self.PI_DLVO(x)*1e9
        integral, err = integrate.quad(func_DLVO, h0, hmax)
        
        integral = integral*1e-9

        
        
        A = (1e6/sigma) * (Pc*h0 - integral)

#        print(f'Integral = {integral}, A = {A}')
        
        if A < 0:
            A = 0
        elif A > 2:
            A = 2
        
        theta = np.arccos(np.abs(1-A)) * 180/np.pi
        
        return(theta)



    def CalculateH0(self):
        """
    		Calculates equilibrium distance h0
    
    		Parameters:
              sigma: Interfacial tension (mN/m)
              Pc: Cappilary pressure (kPa)
              psi1 and psi2: Electrical potential of the plates (V)
              I: Ionic strenght of the solution (mol/kgw)              
    
    
        Returns h0 (nm)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""                  
        Pc = self.Pc
        
        ObjFunction = lambda x0: 1000*np.abs(Pc - self.PI_DLVO(x0[0]*1e-9))
       
        x0 = [0.1, 0]
        bnds = ((0, 2),(0, 1))

#        res = minimize(ObjFunction, x0[0], method='Nelder-Mead', tol=1e-8)
        res = minimize(ObjFunction, x0, method='TNC', bounds=bnds, tol=1e-6)
        
        h0 = res.x[0]
    
        return h0*1e-9


    def DebbyeLength(self):
        
        epsr = self.epsr
        I = self.IonicStrength
        T = self.Temperature
        
        kb = self.PhysicalConstants('kb')
        e = self.PhysicalConstants('e')
        NA = self.PhysicalConstants('NA')
        eps0 = self.PhysicalConstants('eps0')
    
        kappa = (eps0*epsr*kb*T/(2*(e**2)*NA*I))**0.5
        
        return kappa

    def ZetaPotential(self, x, surface_type):
        """
    		Calculates equilibrium distance h0
    
    		Parameters:
              x: distance (m)
              psi_0: Electrical potential of the surface (V)
              I: Ionic strenght of the solution (mol/kgw)              
    
    
        Returns ZetaPotential (V)
        
        Created by Luis Lamas, 23/aug/2021        
     	"""                    
        kb = self.PhysicalConstants('kb')
        e = self.PhysicalConstants('e')
        
        if surface_type == 'Oil':
            psi_0 = self.EDL_Oil
        elif surface_type == 'Calcite':
            psi_0 = self.EDL_Calcite
        T = self.Temperature

        kappa = self.DebbyeLength()
    
        psi_0_reduced = e * psi_0/(kb*T)
        
        gamma = (np.exp(psi_0_reduced/2)-1)/(np.exp(psi_0_reduced/2)+1)
        #gamma = (np.exp(psi_0/2)-1)/(np.exp(psi_0/2)+1)
    
        psi = 2*kb*T/e * np.log((1+gamma*np.exp(-x/kappa))/(1-gamma*np.exp(-x/kappa)))
        #psi = 4*gamma*np.exp(-x/kappa)
        
        return psi    



